In [9]:
import json
import os
import math

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM


# =========================
# Configuration
# =========================

TEACHER_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
STUDENT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

DATA_PATH = "data/distilled.jsonl"
OUTPUT_DIR = "outputs/logit-distilled-student"

TEMPERATURE = 2.0

# Weight of each objective
ALPHA = 0.8   # KL distillation loss
BETA = 0.2    # Cross-entropy loss

LEARNING_RATE = 5e-5

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4

EPOCHS = 1

MAX_LENGTH = 512

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Teacher:", TEACHER_MODEL)
print("Student:", STUDENT_MODEL)

Device: cpu
Teacher: Qwen/Qwen2.5-1.5B-Instruct
Student: Qwen/Qwen2.5-0.5B-Instruct


In [10]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

print("Number of examples:", len(raw_data))

print("\nFirst example:")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False))

Number of examples: 10

First example:
{
  "instruction": "Write a Python function that adds two numbers.",
  "output": "You can define a simple function using two parameters:\n\ndef add(a, b):\n    return a + b\n\nFor example, add(3, 5) returns 8."
}


In [11]:
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Vocabulary size: 151643
PAD token: <|endoftext|>
EOS token: <|im_end|>


In [12]:
example = raw_data[0]

instruction = example["instruction"]
response = example["output"]

messages = [
    {
        "role": "user",
        "content": instruction
    },
    {
        "role": "assistant",
        "content": response
    }
]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Python function that adds two numbers.<|im_end|>
<|im_start|>assistant
You can define a simple function using two parameters:

def add(a, b):
    return a + b

For example, add(3, 5) returns 8.<|im_end|>



In [13]:
encoded = tokenizer(
    formatted_text,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

input_ids = encoded["input_ids"]

print("Input shape:", input_ids.shape)
print("Number of tokens:", input_ids.shape[1])

Input shape: torch.Size([1, 76])
Number of tokens: 76


In [14]:
user_messages = [
    {
        "role": "user",
        "content": instruction
    }
]

assistant_messages = [
    {
        "role": "user",
        "content": instruction
    },
    {
        "role": "assistant",
        "content": response
    }
]

user_text = tokenizer.apply_chat_template(
    user_messages,
    tokenize=False,
    add_generation_prompt=True
)

full_text = tokenizer.apply_chat_template(
    assistant_messages,
    tokenize=False,
    add_generation_prompt=False
)

user_tokens = tokenizer(
    user_text,
    add_special_tokens=False
)["input_ids"]

full_tokens = tokenizer(
    full_text,
    add_special_tokens=False
)["input_ids"]

assistant_start = len(user_tokens)

print("Total tokens:", len(full_tokens))
print("User/prompt tokens:", assistant_start)
print("Assistant tokens:", len(full_tokens) - assistant_start)

Total tokens: 76
User/prompt tokens: 38
Assistant tokens: 38


In [15]:
labels = full_tokens.copy()

for i in range(assistant_start):
    labels[i] = -100

print("Total positions:", len(labels))
print("Ignored positions:", labels.count(-100))
print("Training positions:", sum(x != -100 for x in labels))

Total positions: 76
Ignored positions: 38
Training positions: 38


In [16]:
print("PROMPT:")
print(
    tokenizer.decode(
        full_tokens[:assistant_start],
        skip_special_tokens=False
    )
)

print("\nASSISTANT:")
print(
    tokenizer.decode(
        full_tokens[assistant_start:],
        skip_special_tokens=False
    )
)

PROMPT:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write a Python function that adds two numbers.<|im_end|>
<|im_start|>assistant


ASSISTANT:
You can define a simple function using two parameters:

def add(a, b):
    return a + b

For example, add(3, 5) returns 8.<|im_end|>



In [17]:
class LogitDistillationDataset(Dataset):

    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        example = self.data[idx]

        instruction = example["instruction"]
        response = example["output"]

        user_messages = [
            {
                "role": "user",
                "content": instruction
            }
        ]

        full_messages = [
            {
                "role": "user",
                "content": instruction
            },
            {
                "role": "assistant",
                "content": response
            }
        ]

        # Prompt including assistant generation marker
        user_text = self.tokenizer.apply_chat_template(
            user_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Complete prompt + response
        full_text = self.tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        user_tokens = self.tokenizer(
            user_text,
            add_special_tokens=False
        )["input_ids"]

        full_tokens = self.tokenizer(
            full_text,
            add_special_tokens=False
        )["input_ids"]

        # Truncate
        full_tokens = full_tokens[:self.max_length]

        assistant_start = min(
            len(user_tokens),
            len(full_tokens)
        )

        labels = full_tokens.copy()

        # Ignore prompt tokens
        for i in range(assistant_start):
            labels[i] = -100

        return {
            "input_ids": torch.tensor(
                full_tokens,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


dataset = LogitDistillationDataset(
    raw_data,
    tokenizer,
    MAX_LENGTH
)

print("Dataset size:", len(dataset))

Dataset size: 10


In [18]:
item = dataset[0]

print("Input IDs shape:", item["input_ids"].shape)
print("Labels shape:", item["labels"].shape)

print("\nInput IDs:")
print(item["input_ids"])

print("\nLabels:")
print(item["labels"])

Input IDs shape: torch.Size([76])
Labels shape: torch.Size([76])

Input IDs:
tensor([151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
           553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
            13, 151645,    198, 151644,    872,    198,   7985,    264,  13027,
           729,    429,  11367,   1378,   5109,     13, 151645,    198, 151644,
         77091,    198,   2610,    646,   6979,    264,   4285,    729,   1667,
          1378,   5029,   1447,    750,    912,   2877,     11,    293,    982,
           262,    470,    264,    488,    293,    271,   2461,   3110,     11,
           912,      7,     18,     11,    220,     20,      8,   4675,    220,
            23,     13, 151645,    198])

Labels:
tensor([  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
          -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,  

In [19]:
def collate_fn(batch):

    max_len = max(
        item["input_ids"].size(0)
        for item in batch
    )

    input_ids = []
    labels = []
    attention_masks = []

    for item in batch:

        input_id = item["input_ids"]
        label = item["labels"]

        padding_length = max_len - input_id.size(0)

        input_id = F.pad(
            input_id,
            (0, padding_length),
            value=tokenizer.pad_token_id
        )

        label = F.pad(
            label,
            (0, padding_length),
            value=-100
        )

        attention_mask = (input_id != tokenizer.pad_token_id).long()

        input_ids.append(input_id)
        labels.append(label)
        attention_masks.append(attention_mask)

    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_masks)
    }


dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

batch = next(iter(dataloader))

print("Input IDs:", batch["input_ids"].shape)
print("Labels:", batch["labels"].shape)
print("Attention mask:", batch["attention_mask"].shape)

Input IDs: torch.Size([1, 138])
Labels: torch.Size([1, 138])
Attention mask: torch.Size([1, 138])


In [20]:
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL
)

teacher.to(DEVICE)

teacher.eval()

for param in teacher.parameters():
    param.requires_grad = False

print("Teacher loaded.")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 3425.54it/s]


Teacher loaded.


In [21]:
student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL
)

student.to(DEVICE)

student.train()

print("Student loaded.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2789.80it/s]


Student loaded.


In [22]:
teacher_params = sum(
    p.numel()
    for p in teacher.parameters()
)

student_params = sum(
    p.numel()
    for p in student.parameters()
)

print(f"Teacher parameters: {teacher_params:,}")
print(f"Student parameters: {student_params:,}")

Teacher parameters: 1,543,714,304
Student parameters: 494,032,768


In [23]:
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)

with torch.no_grad():

    teacher_outputs = teacher(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

teacher_logits = teacher_outputs.logits

print("Teacher logits shape:")
print(teacher_logits.shape)

Teacher logits shape:
torch.Size([1, 138, 151936])


In [24]:
student_outputs = student(
    input_ids=input_ids,
    attention_mask=attention_mask
)

student_logits = student_outputs.logits

print("Student logits shape:")
print(student_logits.shape)

Student logits shape:
torch.Size([1, 138, 151936])


In [25]:
shift_teacher_logits = teacher_logits[:, :-1, :]
shift_student_logits = student_logits[:, :-1, :]
shift_labels = batch["labels"][:, 1:].to(DEVICE)

print("Teacher:", shift_teacher_logits.shape)
print("Student:", shift_student_logits.shape)
print("Labels:", shift_labels.shape)

Teacher: torch.Size([1, 137, 151936])
Student: torch.Size([1, 137, 151936])
Labels: torch.Size([1, 137])


In [26]:
loss_mask = shift_labels != -100

print("Total positions:", loss_mask.numel())
print("Positions used for loss:", loss_mask.sum().item())

Total positions: 137
Positions used for loss: 95


In [27]:
teacher_probs = F.softmax(
    shift_teacher_logits / TEMPERATURE,
    dim=-1
)

student_log_probs = F.log_softmax(
    shift_student_logits / TEMPERATURE,
    dim=-1
)

token_kl = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="none"
)

token_kl = token_kl.sum(dim=-1)

masked_kl = token_kl[loss_mask]

kl_loss = masked_kl.mean()

kl_loss = kl_loss * (TEMPERATURE ** 2)

print("KL loss:", kl_loss.item())

KL loss: 1.1640625


In [28]:
student_logits_flat = shift_student_logits.reshape(
    -1,
    shift_student_logits.size(-1)
)

labels_flat = shift_labels.reshape(-1)

ce_loss = F.cross_entropy(
    student_logits_flat,
    labels_flat,
    ignore_index=-100
)

print("Cross-entropy loss:", ce_loss.item())

Cross-entropy loss: 0.984375


In [29]:
total_loss = (
    ALPHA * kl_loss
    + BETA * ce_loss
)

print("KL loss:", kl_loss.item())
print("CE loss:", ce_loss.item())
print("Total loss:", total_loss.item())

KL loss: 1.1640625
CE loss: 0.984375
Total loss: 1.125


In [30]:
optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=LEARNING_RATE
)

print("Optimizer created.")

Optimizer created.


In [31]:
optimizer.zero_grad()

total_loss.backward()

optimizer.step()

print("One training step completed.")

One training step completed.


In [32]:
grad_norm = 0.0

for param in student.parameters():

    if param.grad is not None:
        grad_norm += param.grad.detach().norm().item() ** 2

grad_norm = math.sqrt(grad_norm)

print("Gradient norm:", grad_norm)

Gradient norm: 48.46710360323525


In [33]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

student.train()

for epoch in range(EPOCHS):

    total_epoch_loss = 0.0

    for step, batch in enumerate(dataloader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # -------------------------
        # Teacher forward
        # -------------------------

        with torch.no_grad():

            teacher_outputs = teacher(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            teacher_logits = teacher_outputs.logits

        # -------------------------
        # Student forward
        # -------------------------

        student_outputs = student(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        student_logits = student_outputs.logits

        # -------------------------
        # Shift for causal LM
        # -------------------------

        shift_teacher_logits = teacher_logits[:, :-1, :]
        shift_student_logits = student_logits[:, :-1, :]
        shift_labels = labels[:, 1:]

        loss_mask = shift_labels != -100

        # -------------------------
        # KL divergence
        # -------------------------

        teacher_probs = F.softmax(
            shift_teacher_logits / TEMPERATURE,
            dim=-1
        )

        student_log_probs = F.log_softmax(
            shift_student_logits / TEMPERATURE,
            dim=-1
        )

        token_kl = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="none"
        )

        token_kl = token_kl.sum(dim=-1)

        kl_loss = token_kl[loss_mask].mean()

        kl_loss = kl_loss * (TEMPERATURE ** 2)

        # -------------------------
        # Cross entropy
        # -------------------------

        student_logits_flat = shift_student_logits.reshape(
            -1,
            shift_student_logits.size(-1)
        )

        labels_flat = shift_labels.reshape(-1)

        ce_loss = F.cross_entropy(
            student_logits_flat,
            labels_flat,
            ignore_index=-100
        )

        # -------------------------
        # Combined loss
        # -------------------------

        loss = (
            ALPHA * kl_loss
            + BETA * ce_loss
        )

        # -------------------------
        # Gradient accumulation
        # -------------------------

        loss = loss / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        # -------------------------
        # Optimizer step
        # -------------------------

        if (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or (step + 1) == len(dataloader)
        ):

            optimizer.step()
            optimizer.zero_grad()

        total_epoch_loss += loss.item()

        if step % 2 == 0:

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Step {step + 1}/{len(dataloader)} | "
                f"Loss: {loss.item():.4f} | "
                f"KL: {kl_loss.item():.4f} | "
                f"CE: {ce_loss.item():.4f}"
            )

    avg_loss = total_epoch_loss / len(dataloader)

    print(
        f"\nEpoch {epoch + 1} completed. "
        f"Average loss: {avg_loss:.4f}\n"
    )

Epoch 1/1 | Step 1/10 | Loss: 0.8750 | KL: 4.0000 | CE: 1.5234
Epoch 1/1 | Step 3/10 | Loss: 0.6055 | KL: 2.5938 | CE: 1.7031
Epoch 1/1 | Step 5/10 | Loss: 0.7148 | KL: 3.3281 | CE: 1.0234
Epoch 1/1 | Step 7/10 | Loss: 0.4492 | KL: 1.8047 | CE: 1.7422
Epoch 1/1 | Step 9/10 | Loss: 0.5547 | KL: 2.3750 | CE: 1.6406

Epoch 1 completed. Average loss: 0.6107



In [34]:
del student

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL
)

student.to(DEVICE)
student.train()

optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=LEARNING_RATE
)

print("Student reset and optimizer recreated.")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2959.89it/s]


Student reset and optimizer recreated.


In [35]:
student.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Student model saved to:")
print(OUTPUT_DIR)

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.61s/it]

Student model saved to:
outputs/logit-distilled-student
